In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
def arbol_crr(S0, K, r, T, sigma, N, option='call', style='european'):

    dt = T / N
    u  = np.exp(sigma * np.sqrt(dt))
    d  = 1 / u
    p  = (np.exp(r * dt) - d) / (u - d)
    df = np.exp(-r * dt)   # factor de descuento por paso

    # --- Árbol de precios al vencimiento ---
    # ST[i] = S0 * u^(N-i) * d^i  para i = 0, 1, ..., N
    ST = np.array([S0 * (u**(N-i)) * (d**i) for i in range(N+1)])

    # --- Valores de la opción al vencimiento ---
    if option == 'call':
        V = np.maximum(ST - K, 0)
    else:
        V = np.maximum(K - ST, 0)

    # --- Inducción hacia atrás ---
    for step in range(N-1, -1, -1):
        S_step = np.array([S0 * (u**(step-i)) * (d**i) for i in range(step+1)])

        # Valor de continuar (esperanza descontada)
        V_continuar = df * (p * V[:step+1] + (1-p) * V[1:step+2])

        if style == 'american':
            # Valor de ejercer ahora
            if option == 'call':
                V_ejercer = np.maximum(S_step - K, 0)
            else:
                V_ejercer = np.maximum(K - S_step, 0)
            V = np.maximum(V_continuar, V_ejercer)
        else:
            V = V_continuar

    return V[0]

# Parámetros
S0    = 669.03
K     = 680
r     = 0.042
T     = 31/365
sigma = 0.1687
N     = 100

precio_euro_call = arbol_crr(S0, K, r, T, sigma, N, 'call', 'european')
precio_amer_call = arbol_crr(S0, K, r, T, sigma, N, 'call', 'american')
precio_euro_put  = arbol_crr(S0, K, r, T, sigma, N, 'put',  'european')
precio_amer_put  = arbol_crr(S0, K, r, T, sigma, N, 'put',  'american')

print(f"Call europea  : ${precio_euro_call:.4f}")
print(f"Call americana: ${precio_amer_call:.4f}  (prima por ejercicio anticipado: ${precio_amer_call - precio_euro_call:.4f})")
print(f"Put europea   : ${precio_euro_put:.4f}")
print(f"Put americana : ${precio_amer_put:.4f}  (prima por ejercicio anticipado: ${precio_amer_put - precio_euro_put:.4f})")

Call europea  : $9.3925
Call americana: $9.3925  (prima por ejercicio anticipado: $0.0000)
Put europea   : $17.9412
Put americana : $18.2479  (prima por ejercicio anticipado: $0.3068)


¿Por qué el ejercicio anticipado de una call sobre un activo sin dividendos nunca es óptimo? ¿Qué cambia si el activo paga dividendos? Bueno pues si ejerces antes  pagas el strike  (K) de hoy , ta,bien pierdes el seguro de que el precio pueda suir . de lo contario si no ejerces  pues mantienes la opcion viva  y puedes invertir ese dinero. Siempre  es mejor vender la opcion o mantenerla, pero no ejercer.
Si la acción paga dividendos:El precio de la acción suele bajar después del dividendo

En la gráfica de convergencia, ¿por qué el precio oscila antes de converger? ¿Qué relación tiene eso con si N es par o impar?l modelo binomial es una aproximación discreta.Aumentas N y te acercas al valor real (tipo Black-Scholes)
Pero no lo hace de forma suave, sino en zig-zag



La prima por ejercicio anticipado es mayor para puts muy ITM. ¿Por qué intuitivamente tiene sentido ejercer anticipadamente una put muy ITM?
Una put muy ITM significa:Precio actual S muy por debajo de K
Ya tienes una ganancia grande asegurada: K−S
El precio podría subir después (riesgo de perder valor)
Ademmas, Ejerciendo antes  recibes el dinero ya
Puedes invertirlo inmediatamente, entonces si cobras antes +  reduces el riesgo eso tiene sentido a ejercer, mientras no.


Si modificas T primero a 31 días y luego a 1 año, ¿cómo esperas que cambie la diferencia entre la put americana y la europea? ¿Por qué? si aumentas  tiempo  en 31 dias , poca diferencia entre pu americana y europea, con un año la diferencia aumenta. esto porqyue hay mas oportunidades de que el precio caiga mucho  y que convenga ejercer antes.


El árbol CRR usa probabilidades neutrales al riesgo p, no probabilidades reales. ¿Qué significa eso y por qué es válido para el pricing? Las probabilidades neutrales del riesgo no son probabilidades reales del mercado  si no que son ajustadas  para que con esto todo los activos cresca al tipo libre de riesgo.